# TROPOMI solar-induced fluorescence (SIF) — Ogallala bounding box, native soundings, one CF file per product
Provider's catalogue → verified downloads → every variable decoded (corruption check) → soundings of the aquifer's
bounding box picked out by 4 dask workers → appended to a CF NetCDF → read back and compared → source files deleted →
repeat. Hourly verified copy to Drive, resumable from the Drive file. The structure, checks and no-manipulation rules are
those of `OCO_SIF_Lite_Ogallala.ipynb`.

**Data** (two TROPOMI SIF products, two output files; set `PRODUCTS_TO_RUN` to run one):

| | `TROPOSIF` | `CALTECH` |
|---|---|---|
| Product | ESA **TROPOSIF L2B**, version 2.1 (Guanter et al. 2021) | **Caltech TROPOMI SIF 740 nm**, ungridded (Koehler et al. 2018, 2020) |
| Source | `https://ftp.sron.nl/open-access-data-2/TROPOMI/tropomi/sif/v2.1/l2b/` (plain HTTPS) | `ftp://fluo.gps.caltech.edu/data/tropomi/ungridded/SIF740nm/` (anonymous FTP) |
| Files | one daily file per day, only valid retrievals, about 195 MB, 2.9 million soundings | one monthly `.tar.gz` of daily files, about 1–3 GB each |
| Record (listed 2026-10-22) | 2018-05 → 2021-12, 1,186 daily files, about 213 GB (some days have no file, e.g. 2019-06-15) | 2017-11 → 2022-12, 60 monthly archives, about 169 GB |
| Output (`MyDrive/MSUGWB/TROPOMI_SIF/`) | `TROPOSIF_L2B_v2.1_Ogallala.nc` | `TROPOMI_SIF740_Caltech_Ogallala.nc` |

Neither product has a spatial index, so every file is downloaded and filtered here. Not included: Caltech's red SIF at 683 nm and
both providers' gridded products (the task is native soundings).

**Output** — each file has two record axes:

| Axis | One row per | Holds |
|---|---|---|
| `sounding` | sounding kept, in time order | every per-sounding variable the provider publishes (TROPOSIF: SIF at 743–758 nm and 735–758 nm, day-length-corrected SIF, 1-sigma errors, TOA reflectance at 7 wavelengths, mean radiances, angles, footprint corners, L2 cloud fraction, land cover; Caltech: SIF, error, relative SIF, day-length-corrected SIF, VIIRS cloud fraction, angles, daily correction factor, footprint corners, NIR radiance) |
| `day` | daily file processed, also when it has no sounding here | the provider's per-file tables (TROPOSIF: reference time, reflectance wavelengths, retrieval settings as JSON, creation date) and the ledger: file, item, size, ETag / modification time, counts |

Added per sounding: `day_index`, `source_index` (position in the provider's file), `in_polygon` (centre inside the aquifer polygon),
`footprint_in_polygon` (footprint touches it), and for TROPOSIF `sounding_time` (see below).

**No-manipulation design**
- A sounding is kept when its centre, or any footprint corner, lies in the aquifer's bounding box. Nothing else is filtered: no
  quality, cloud or angle screening. The providers' own screening (TROPOSIF L2B holds only retrievals that passed it) is theirs.
- No gridding, averaging, gap filling, scaling or masking. Every value is the value in the provider's file, in its stored type,
  missing-value codes included. Variables are stored under their path in the file with the group folded into the name
  (`PRODUCT/SIF_743` → `SIF_743`); `source_variable` holds the original path. The provider's `valid_range` is kept as `source_valid_range`.
- The Caltech corner arrays store the sounding axis last; it is moved to the front (`source_dimensions` says so). Nothing else changes.
- Variables are recognised by their axes, not by name, so one that a later release adds is carried along.
- **Damaged soundings are the one exception.** A provider's file can hold rows whose columns no longer belong together: Caltech's
  `TROPO_SIF_2020-02-13_ungridded.nc` has 109,267 contiguous rows (5.5 %) with times in 2009-2010 and centres about 25° from their own footprint
  corners. A sounding whose time lies outside its file's day (more than a day before or two days after it), or whose centre is farther than
  `POSITION_TOL_DEG` (1°; healthy files: 0.07°) from one of its own corners, cannot be placed, so it is not stored. They are counted for every day in
  `day_n_damaged_soundings`, listed in `damaged_soundings.jsonl` next to the file, and reported in the log. A file in which more than
  `MAX_DAMAGED_FRACTION` (25 %) of the soundings are damaged is not stored and stops the run. Days stored before this test existed hold -1 in that column.
- Cell 9 downloads a few stored files again and compares every stored value with the provider's file, bit for bit.

**Things to know about the files** (measured on real files on 2026-10-22, not changed here)
- **TROPOSIF** declares no missing-value code, yet its 735–758 nm fields (`SIF_735`, `SIF_Corr_735`, `SIF_ERROR_735`, `Mean_TOA_RAD_735`)
  hold the netCDF default 9.96921e36 for soundings that window could not fit (about 0.4 % of the soundings kept). That value is
  declared as `_FillValue` here and says so in the variable's `comment`. The 743–758 nm fields had none.
- TROPOSIF's `standard_name` attributes are free text (for example "retrieved SIF@740 743-758 nm fitting window"), not CF names; they are kept
  as `source_standard_name`. Its latitude, longitude and angle variables state no units; degrees are given from their names and values.
- TROPOSIF stores the time of a sounding as `delta_time`, milliseconds since 00:00 UTC of the file's day (the units text names that day).
  `sounding_time` is the same instant in seconds since 1970-01-01, computed exactly; the scalar `reference_time` is a float32 and too coarse for this.
- Caltech's soundings are not in time order inside a daily file; they are sorted here (stable). Its files carry no attributes besides
  variable units, long names and `_FillValue = -999` (which did not occur in the day inspected).
- Caltech's archives hold the days of their month only; `day_source_item` is the archive, `day_source_file` the daily file.
- The providers ask for acknowledgement: TROPOSIF has a data use policy (`TROPOSIF_Data_Use_Policy.pdf` on the server); Caltech's `READ_ME.txt` asks to be informed of any use in a publication.
- Rehearsal speed from a home connection: SRON about 20 MB/s, Caltech's FTP about 3 MB/s, so the Caltech record is the long part of a full run.

**Reading the files**
- `xarray.open_dataset(path)` works; it turns missing-value codes into NaN and integer variables that have a code into floats (use
  `mask_and_scale=False` for the stored integers).
- Stop at `committed_soundings` / `committed_days` if `processing_status` is not `complete`.
- Typical filter: `in_polygon == 1`, plus a choice of cloud fraction and solar zenith angle (none is applied here).

**Before running**
1. `high_plains_quifer.zip` in `MyDrive/`. No login is needed for either source.
2. `TEST_MODE = True` first: 6 TROPOSIF days and 7 Caltech days into `.../_test/`, then a projection of the full run (from the
   rehearsal: TROPOSIF roughly 3-4 h and 1.6 GB, Caltech roughly 15-19 h at the speed measured and 1.3 GB; the time is mostly download).
   Then set `TEST_MODE = False` and Run all. An interrupted run continues from the file on Drive; run it again later to add new days.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# numpy, geopandas, shapely and requests are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q netCDF4 "dask[distributed]" 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE  = True              # True -> short rehearsal into .../_test/ ; False -> the whole record
TEST_DATES = {"TROPOSIF": ("2019-06-10", "2019-06-17"),    # rehearsal window per product (TROPOSIF: 8 daily files of
              "CALTECH":  ("2017-11-23", "2017-11-30")}    # about 200 MB; Caltech: the smallest monthly archive, 7 days)

PRODUCTS_TO_RUN = ["TROPOSIF", "CALTECH"]   # one output file per product: ESA's TROPOSIF L2B and Caltech's TROPOMI SIF
DATE_START = None              # None -> first day the provider holds; or "YYYY-MM-DD"
DATE_END   = None              # None -> latest day the provider holds; or "YYYY-MM-DD"
ROI_PAD_DEG = 0.0              # widen the aquifer's bounding box by this many degrees on every side

N_DOWNLOAD_THREADS = 6         # files (TROPOSIF) or archives (Caltech) downloaded at a time
DASK_WORKERS       = 4         # worker processes that verify the files and pick out the aquifer's soundings
WORKER_MEMORY_GB   = 2.5       # hard memory limit per dask worker (a global TROPOSIF day holds 2.9 million soundings)
BATCH_ITEMS        = {"TROPOSIF": 20,   # daily files per cycle: download, extract, write, verify, delete (about 4 GB)
                      "CALTECH": 2}     # monthly archives per cycle (about 6 GB, about 50 daily files)
COMPRESS_LEVEL     = 1         # zlib level of the CF .nc (lossless)
SOUNDING_CHUNK     = 32768     # soundings per HDF5 chunk of the output
MAX_DAMAGED_FRACTION = 0.25    # a daily file in which more than this share of all soundings is damaged (Cell 6) is not stored and stops the run
POSITION_TOL_DEG     = 1.0     # a sounding is damaged when its centre is farther than this from one of its own footprint corners
                               # (healthy files: at most 0.07 degrees)

REPROCESS_CHANGED = False      # True -> rebuild from the first stored file the provider has replaced or added
REBUILD           = False      # True -> ignore the files on Drive and start the outputs again

MIN_FREE_GB       = 20.0       # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600   # give up (cleanly, resumable) if space never comes back
BACKUP_INTERVAL_S = 3600       # copy the file in progress to Drive this often
MAX_RETRIES       = 4          # extra download attempts per file
CROSS_CHECKS      = {"TROPOSIF": 4,    # stored files per product that are downloaded again and compared value by value
                     "CALTECH": 1}     # (one archive of about 1-3 GB)
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, the two products, CF wording ═══════════════
import re, gc, sys, json, time, shutil, hashlib, logging, threading, random, tarfile, ftplib
from datetime import datetime, timezone, timedelta, date
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import shapely
import requests
import dask
from dask.distributed import Client, LocalCluster

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a
# no-op there and INFO progress lines would never be shown.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("tropomisif")
for _n in ("distributed", "urllib3.connectionpool", "pyogrio", "fiona"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL  = "Ogallala"
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT  = f"{DRIVE_ROOT}/TROPOMI_SIF" + ("/_test" if TEST_MODE else "")
SCRATCH    = "/content/tropomisif_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR    = f"{SCRATCH}/raw"            # downloaded daily files / archives (deleted after use)
NC_DIR     = f"{SCRATCH}/nc"             # the output files being built
DASK_TMP   = f"{SCRATCH}/dask"
DISK_PATH  = "/content"                  # same filesystem as scratch AND Drive's local upload cache
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, DASK_TMP):
    os.makedirs(_d, exist_ok=True)

# ---------- what the providers publish ----------
# Both products are daily global files of single soundings (footprints) with no spatial index, so every file is
# downloaded and the aquifer's soundings are picked out here, as in OCO_SIF_Lite_Ogallala.ipynb.
PRODUCTS = {
    "TROPOSIF": dict(
        label="ESA TROPOSIF L2B, version 2.1", platform="Sentinel-5P TROPOMI", out_name=f"TROPOSIF_L2B_v2.1_{ROI_LABEL}.nc",
        source="sron", base="https://ftp.sron.nl/open-access-data-2/TROPOMI/tropomi/sif/v2.1/l2b/",
        file_re=re.compile(r"TROPOSIF_L2B_(\d{4})-(\d{2})-(\d{2})\.nc$"),
        institution="Data: TROPOSIF, Guanter et al. (Universitat Politecnica de Valencia / Noveltis), ESA S5P+Innovation project, "
                    "served by SRON. Subset: MSU groundwater project.",
        references="Guanter et al. 2021, ESSD 13, 5423, https://doi.org/10.5194/essd-13-5423-2021 ; https://s5p-troposif.noveltis.fr ; "
                   "data use policy: TROPOSIF_Data_Use_Policy.pdf next to the files",
        sound_dim="n_elem", lat="PRODUCT/latitude", lon="PRODUCT/longitude",
        clat="PRODUCT/SUPPORT_DATA/GEOLOCATIONS/latitude_bounds", clon="PRODUCT/SUPPORT_DATA/GEOLOCATIONS/longitude_bounds",
        time=dict(kind="delta", path="PRODUCT/delta_time"),         # milliseconds since 00:00 UTC of the file's day
        name_strip=("PRODUCT/SUPPORT_DATA/", "PRODUCT/"),
        rename={"PRODUCT/time": "reference_time"},                  # a variable called "time" next to the day axis would confuse readers
        units_forced={"PRODUCT/latitude": "degrees_north", "PRODUCT/longitude": "degrees_east",
                      "PRODUCT/SUPPORT_DATA/GEOLOCATIONS/viewing_zenith_angle": "degree",
                      "PRODUCT/SUPPORT_DATA/GEOLOCATIONS/solar_zenith_angle": "degree",
                      "PRODUCT/SUPPORT_DATA/GEOLOCATIONS/relative_azimuth_angle": "degree"},   # the file states no units; degrees by name and range
        globals_per_day=["date_created"], globals_skip=[],
        day_json_group="METADATA/ALGORITHM_SETTINGS",               # the retrieval settings, kept per day as JSON
        undeclared_fill=True),                                      # the files hold the netCDF default fill without declaring it
    "CALTECH": dict(
        label="Caltech TROPOMI SIF 740 nm (Koehler et al.), ungridded", platform="Sentinel-5P TROPOMI",
        out_name=f"TROPOMI_SIF740_Caltech_{ROI_LABEL}.nc",
        source="caltech", host="fluo.gps.caltech.edu", folder="/data/tropomi/ungridded/SIF740nm/",
        file_re=re.compile(r"TROPO_SIF_(\d{4})-(\d{2})\.tar\.gz$"),
        member_re=re.compile(r"TROPO_SIF_(\d{4})-(\d{2})-(\d{2})_ungridded\.nc$"),
        institution="Data: Caltech (Philipp Koehler, Christian Frankenberg), fluo.gps.caltech.edu. Subset: MSU groundwater project.",
        references="Koehler et al. 2018, GRL 45, 10456, https://doi.org/10.1029/2018GL079031 ; Koehler et al. 2020 ; "
                   "ftp://fluo.gps.caltech.edu/data/tropomi/READ_ME.txt asks users to inform the authors of any publication use",
        sound_dim="TIME", lat="lat", lon="lon", clat="lat_bnds", clon="lon_bnds",
        time=dict(kind="direct", path="TIME"),                      # seconds since 1970-01-01 UTC
        name_strip=(), rename={}, units_forced={}, globals_per_day=[], globals_skip=[], day_json_group=None, undeclared_fill=False),
}
CF_VERSION = "CF-1.9"                    # first CF version that allows 64-bit integers and string variables
DAY_EPOCH  = date(1970, 1, 1)
NAME_ADDED = "sounding_time"             # the one variable computed here (TROPOSIF): exact UTC time of every sounding


def nc_name(spec, path):
    """Output name of one of the provider's variables: its path with the group folded in and the folders
    PRODUCT/ and PRODUCT/SUPPORT_DATA/ left out, e.g. PRODUCT/SIF_743 -> SIF_743 and
    PRODUCT/SUPPORT_DATA/GEOLOCATIONS/solar_zenith_angle -> GEOLOCATIONS_solar_zenith_angle.
    source_variable holds the original path, so the name is reversible; a collision stops the run."""
    if path in spec["rename"]:
        return spec["rename"][path]
    for pre in spec["name_strip"]:
        if path.startswith(pre):
            path = path[len(pre):]
            break
    return path.replace("/", "_")


# ---------- CF wording. Descriptive only: no stored value is touched ----------
UNITS_CF = {"mW/m2/sr/nm": "mW m-2 sr-1 nm-1", "mW/m^2/sr/nm": "mW m-2 sr-1 nm-1", "degrees": "degree", "-": "1", "%": "percent",
            "degrees_north": "degrees_north", "degrees_east": "degrees_east", "nm": "nm",
            "UTC, seconds since 1970-01-01 00:00:00": "seconds since 1970-01-01 00:00:00"}
STANDARD_NAME_OK = {"latitude", "longitude", "time", "solar_zenith_angle", "sensor_zenith_angle"}   # others in the files are free text
DELTA_COMMENT = ("Copied unchanged: milliseconds since 00:00:00 UTC of the day of the file (the source's units text names that day, "
                 "which is different in every file; day_time holds it). sounding_time is the same instant as seconds since "
                 "1970-01-01 UTC, computed here exactly from the two.")


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_day(s):
    return datetime.strptime(s, "%Y-%m-%d").date()


def day_number(d):
    return (d - DAY_EPOCH).days


UNKNOWN = [p for p in PRODUCTS_TO_RUN if p not in PRODUCTS]
if UNKNOWN:
    raise ValueError(f"PRODUCTS_TO_RUN names {UNKNOWN}; the notebook knows {sorted(PRODUCTS)}.")
log.info("products: %s | TEST_MODE=%s", " ".join(PRODUCTS_TO_RUN), TEST_MODE)

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: the providers' file lists, verified downloads, the aquifer polygon ═══════════════
# An "item" is what is downloaded: one daily file (TROPOSIF) or one monthly archive of daily files (Caltech).
# A "gran" is one daily file on scratch, ready to be read.
_tls = threading.local()


def _http(method, url, **kw):
    """One request with retries; the providers' servers are public, no login is sent."""
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            r = requests.request(method, url, timeout=(30, 120), allow_redirects=True, **kw)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{url} did not answer: {last}. Nothing is lost - re-run to resume.")


def window(product):
    """(first day, last day) of this run; None = no limit."""
    a, b = TEST_DATES[product] if TEST_MODE else (DATE_START, DATE_END)
    return (parse_day(a) if a else None), (parse_day(b) if b else None)


def _in_window(d, lo, hi):
    return (lo is None or d >= lo) and (hi is None or d <= hi)


# ---------- TROPOSIF at SRON: <base>/<year>/<month>/TROPOSIF_L2B_<date>.nc ----------
def sron_items(spec, lo, hi, heads=True):
    index = _http("GET", spec["base"]).text
    years = sorted(int(y) for y in re.findall(r'href="(\d{4})/"', index))
    out = []
    for y in years:
        if (lo and y < lo.year) or (hi and y > hi.year):
            continue
        months = sorted(int(m) for m in re.findall(r'href="(\d{2})/"', _http("GET", f"{spec['base']}{y}/").text))
        for m in months:
            if (lo and (y, m) < (lo.year, lo.month)) or (hi and (y, m) > (hi.year, hi.month)):
                continue
            page = _http("GET", f"{spec['base']}{y}/{m:02d}/").text
            for name in sorted(set(re.findall(r'href="(TROPOSIF_L2B_[\d-]+\.nc)"', page))):
                mt = spec["file_re"].search(name)
                if not mt:
                    continue
                day = date(int(mt.group(1)), int(mt.group(2)), int(mt.group(3)))
                if _in_window(day, lo, hi):
                    out.append(dict(name=name, key=name, day=day, url=f"{spec['base']}{y}/{m:02d}/{name}"))

    if not heads:
        return out

    def head(it):
        r = _http("HEAD", it["url"])
        it.update(size=int(r.headers["Content-Length"]), etag=(r.headers.get("ETag") or "").strip('"'),
                  last_modified=r.headers.get("Last-Modified", ""))
        return it

    with ThreadPoolExecutor(6) as ex:
        return list(ex.map(head, out))


# ---------- Caltech: FTP, one tar.gz per month ----------
def _ftp(spec):
    ftp = ftplib.FTP(spec["host"], timeout=120)
    ftp.login()                                                # anonymous
    ftp.voidcmd("TYPE I")
    ftp.cwd(spec["folder"])
    return ftp


def caltech_items(spec, lo, hi):
    out = []
    ftp = _ftp(spec)
    try:
        for name in sorted(ftp.nlst()):
            mt = spec["file_re"].search(name)
            if not mt:
                continue
            y, m = int(mt.group(1)), int(mt.group(2))
            first, last = date(y, m, 1), (date(y + (m == 12), m % 12 + 1, 1) - timedelta(days=1))
            if (lo and last < lo) or (hi and first > hi):
                continue
            out.append(dict(name=name, key=name, day=first, size=int(ftp.size(name)),
                            last_modified=ftp.voidcmd("MDTM " + name)[4:].strip(), etag=""))
    finally:
        ftp.close()
    return out


def select_items(product, whole_record=False):
    """The items to have in the output, in date order, and the window used (whole_record: no window at all)."""
    spec = PRODUCTS[product]
    lo, hi = (None, None) if whole_record else window(product)
    items = sron_items(spec, lo, hi, heads=not whole_record) if spec["source"] == "sron" else caltech_items(spec, lo, hi)
    items.sort(key=lambda it: it["name"])
    if not items:
        raise PipelineStop(f"{product}: the provider lists nothing between {lo} and {hi}.")
    dup = [k for k in {it["key"] for it in items} if sum(1 for it in items if it["key"] == k) > 1]
    if dup:
        raise DataIntegrityError(f"{product}: the listing names {dup[:3]} twice.")
    return items, lo, hi


def calendar_gaps(days):
    """Runs of calendar days without a file: [(first, last, length), ...]."""
    return [(a + timedelta(days=1), b - timedelta(days=1), (b - a).days - 1)
            for a, b in zip(days, days[1:]) if (b - a).days > 1]


# ---------- downloads ----------
def _check_complete(path, expected):
    got = os.path.getsize(path)
    if expected is not None and got != int(expected):
        raise IOError(f"incomplete: {got} of {expected} bytes")
    with open(path, "rb") as f:
        magic = f.read(4)
    return got, magic


def download_http(spec, it, raw_dir):
    """One daily TROPOSIF file: byte count equals the server's Content-Length, the file starts like an HDF5 file
    and its MD5 equals the server's ETag when that is an MD5. Every variable is decoded afterwards by a worker."""
    dest = os.path.join(raw_dir, it["name"])
    part = dest + ".part"
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            with requests.get(it["url"], stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if "html" in (r.headers.get("Content-Type") or "").lower():
                    raise IOError("the server sent a web page, not data")
                expected = r.headers.get("Content-Length")
                etag = (r.headers.get("ETag") or "").strip('"')
                modified = r.headers.get("Last-Modified", "")
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
                    f.flush()
                    os.fsync(f.fileno())
            got, magic = _check_complete(part, expected)
            if magic != b"\x89HDF":
                raise IOError(f"not an HDF5 / NetCDF-4 file (starts with {magic!r})")
            if re.fullmatch(r"[0-9a-f]{32}", etag) and md5sum(part) != etag:
                raise IOError("MD5 differs from the server's")
            os.replace(part, dest)
            return dict(size=got, etag=etag, last_modified=modified)
        except Exception as e:
            last = e
            try:
                os.remove(part)
            except OSError:
                pass
            log.warning("download %s attempt %d/%d: %s", it["name"], attempt, MAX_RETRIES + 1, e)
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{it['name']} could not be downloaded after {MAX_RETRIES + 1} attempts ({last}). Nothing is lost - re-run to resume.")


def download_ftp(spec, it, raw_dir):
    """One monthly Caltech archive, continued from the bytes already on scratch after a break. The server gives no
    checksum: the byte count must equal SIZE, and extracting reads the whole gzip stream (its CRC is checked)."""
    dest = os.path.join(raw_dir, it["name"])
    part = dest + ".part"
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            ftp = _ftp(spec)
            try:
                have = os.path.getsize(part) if os.path.exists(part) else 0
                with open(part, "ab" if have else "wb") as f:
                    ftp.retrbinary("RETR " + it["name"], f.write, blocksize=1 << 20, rest=have or None)
                    f.flush()
                    os.fsync(f.fileno())
            finally:
                try:
                    ftp.close()
                except Exception:
                    pass
            if os.path.getsize(part) != it["size"]:
                raise IOError(f"incomplete: {os.path.getsize(part)} of {it['size']} bytes")
            os.replace(part, dest)
            return dest
        except Exception as e:
            last = e
            log.warning("download %s attempt %d/%d: %s", it["name"], attempt, MAX_RETRIES + 1, e)
            time.sleep(min(60, 10 * attempt))
    raise PipelineStop(f"{it['name']} could not be downloaded after {MAX_RETRIES + 1} attempts ({last}). Nothing is lost - re-run to resume.")


def extract_tar(spec, it, tar_path, raw_dir, lo, hi):
    """Take the daily files out of a monthly archive (inside the date window). Returns the grans."""
    grans = []
    try:
        with tarfile.open(tar_path, "r:gz") as tf:
            members = sorted((m for m in tf.getmembers() if m.isfile()), key=lambda m: m.name)
            for m in members:
                base = os.path.basename(m.name)
                mt = spec["member_re"].search(base)
                if not mt:
                    log.warning("%s holds %s, which is not a daily file name - ignored", it["name"], m.name)
                    continue
                day = date(int(mt.group(1)), int(mt.group(2)), int(mt.group(3)))
                if not _in_window(day, lo, hi):
                    continue
                dest = os.path.join(raw_dir, base)
                with tf.extractfile(m) as src, open(dest + ".part", "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 20)
                if os.path.getsize(dest + ".part") != m.size:
                    raise IOError(f"{base}: {os.path.getsize(dest + '.part')} of {m.size} bytes extracted")
                os.replace(dest + ".part", dest)
                grans.append(dict(name=base, day=day, size=m.size))
    except (tarfile.TarError, EOFError, OSError) as e:
        raise IOError(f"{it['name']} cannot be read as a gzip tar archive ({e})")
    return grans


def fetch_item(product, it, raw_dir=None):
    """Download one item and return its daily files on scratch: [dict(name, day, item, size, etag, last_modified, ...)]."""
    spec = PRODUCTS[product]
    raw_dir = raw_dir or RAW_DIR
    lo, hi = window(product)
    if spec["source"] == "sron":
        info = download_http(spec, it, raw_dir)
        return [dict(name=it["name"], day=it["day"], item=it["key"], size=info["size"], etag=info["etag"],
                     last_modified=info["last_modified"], item_size=info["size"], item_last_modified=info["last_modified"])]
    for attempt in (1, 2):                                       # a damaged archive is fetched once more
        tar_path = download_ftp(spec, it, raw_dir)
        try:
            grans = extract_tar(spec, it, tar_path, raw_dir, lo, hi)
            break
        except IOError as e:
            os.remove(tar_path)
            if attempt == 2:
                raise PipelineStop(f"{e} after two downloads. Nothing is lost - re-run to resume; if it persists the archive is damaged at the source.")
            log.warning("%s - downloading it again", e)
    os.remove(tar_path)
    for g in grans:
        g.update(item=it["key"], etag="", last_modified=it["last_modified"], item_size=it["size"], item_last_modified=it["last_modified"])
    return grans


def fetch_batch(product, items):
    """Download a batch, N_DOWNLOAD_THREADS at a time, never eating into the scratch floor. Returns the daily files in
    date order. A day found in two items stops the run (a day must be stored once)."""
    with timed("download"):
        spec = PRODUCTS[product]
        per_item = 0.25 if spec["source"] == "sron" else 3.5
        ensure_disk_free("download", extra_gb=len(items) * per_item)
        t0 = time.time()
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            lists = list(ex.map(lambda it: fetch_item(product, it), items))
        grans = sorted((g for l in lists for g in l), key=lambda g: (g["day"], g["name"]))
        days = [g["day"] for g in grans]
        if len(set(days)) != len(days):
            raise DataIntegrityError(f"{product}: a day occurs in more than one daily file of this batch: "
                                     f"{sorted({d for d in days if days.count(d) > 1})[:3]}")
        size = sum(it["size"] for it in items)
        dt = max(1e-6, time.time() - t0)
        log.info("Downloaded %d item(s) = %d daily file(s), %s in %.0f s (%.1f MB/s)", len(items), len(grans), human(size), dt, size / 1e6 / dt)
        TOTALS["bytes_downloaded"] = TOTALS.get("bytes_downloaded", 0) + size
    return grans


def remove_raw(names, raw_dir=None):
    for n in names:
        for p in (os.path.join(raw_dir or RAW_DIR, n), os.path.join(raw_dir or RAW_DIR, n + ".part")):
            try:
                os.remove(p)
            except OSError:
                pass


# ---------- the aquifer polygon ----------
def load_roi():
    """The aquifer polygon (EPSG:4326, not simplified) and the bounding box whose soundings are kept."""
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise DataIntegrityError("The aquifer shapefile has no CRS.")
    g4 = gdf.to_crs(epsg=4326).geometry
    poly = g4.union_all() if hasattr(g4, "union_all") else g4.unary_union
    if not poly.is_valid:
        poly = poly.buffer(0)
    w, s, e, n = poly.bounds
    bbox = (w - ROI_PAD_DEG, s - ROI_PAD_DEG, e + ROI_PAD_DEG, n + ROI_PAD_DEG)
    log.info("Aquifer polygon: %d vertices | bounding box kept: lon %.4f to %.4f, lat %.4f to %.4f",
             shapely.get_num_coordinates(poly), bbox[0], bbox[2], bbox[1], bbox[3])
    return dict(poly=poly, bbox=tuple(float(v) for v in bbox), wkb=shapely.to_wkb(poly))

In [ ]:
# ═══════════════ CELL 6: verify one daily file and pick out the aquifer's soundings (runs in dask workers) ═══════════════
def extract_file(path, spec, roi_wkb, day0=None):
    """Decode EVERY variable of one daily file (a damaged file fails here), then keep the soundings whose centre or
    any footprint corner lies in the bounding box, except damaged ones: a sounding whose time lies more than a day before or
    two days after the start of the file's day (day0, seconds since 1970), or whose centre is farther than spec["pos_tol"]
    degrees from one of its own footprint corners, cannot be placed in time or space. They are counted, not stored. Values are returned exactly as stored: no scaling, masking,
    casting or filtering. Returns plain numpy / Python objects.

    A variable is a "sounding" variable when the per-sounding axis is one of its axes (in the Caltech files it is the
    last axis of the corner arrays; it is moved to the front, nothing else changes), a "day" variable otherwise
    (per-file tables and labels; axes of length 1 are dropped). Variables are recognised by their axes, not by name, so
    one that a later release adds is carried along without a change here."""
    import os, re, json
    import numpy as np
    import netCDF4 as nc4
    import shapely
    from datetime import datetime

    out = dict(name=os.path.basename(path), ok=False)
    try:
        def plain(v):
            if isinstance(v, np.ndarray):
                return v.tolist() if v.size != 1 else v.reshape(-1)[0].item()
            if isinstance(v, np.generic):
                return v.item()
            return v

        def walk(g, prefix=""):
            for n, v in g.variables.items():
                yield prefix + n, v
            for n, sub in g.groups.items():
                yield from walk(sub, prefix + n + "/")

        sd = spec["sound_dim"]
        meta, sounding, daily = {}, {}, {}
        with nc4.Dataset(path) as ds:
            ds.set_auto_maskandscale(False)                      # raw stored values, in every group
            if sd not in ds.dimensions:
                raise ValueError("the file has no per-sounding axis")
            n_file = ds.dimensions[sd].size
            dim_sizes = {k: int(d.size) for k, d in ds.dimensions.items()}
            glob = {k: str(ds.getncattr(k)) for k in ds.ncattrs() if k not in spec["globals_skip"]}
            day_json = None
            if spec["day_json_group"]:
                g = ds
                for part in spec["day_json_group"].split("/"):
                    g = g.groups[part]
                day_json = json.dumps({k.strip(): plain(g.getncattr(k)) for k in g.ncattrs()}, sort_keys=True)

            def read(p, v):
                a = v[...]                                       # full read = decode check of every chunk
                a = np.asarray(a)
                if a.dtype != v.dtype:
                    raise ValueError(f"{p} was not read in its stored type")
                return a

            root = ds
            def var(p):
                g = root
                *groups, n = p.split("/")
                for part in groups:
                    g = g.groups[part]
                return g.variables[n]

            # position first: the polygon test needs only latitude, longitude and the corners
            lat, lon = read(spec["lat"], var(spec["lat"])), read(spec["lon"], var(spec["lon"]))
            if lat.shape != (n_file,) or lon.shape != (n_file,):
                raise ValueError("latitude / longitude are not one value per sounding")
            cv = var(spec["clat"])
            cpos = list(cv.dimensions).index(sd)
            clat = np.moveaxis(read(spec["clat"], cv), cpos, 0)
            clon = np.moveaxis(read(spec["clon"], var(spec["clon"])), list(var(spec["clon"]).dimensions).index(sd), 0)
            w, s, e, n = spec["bbox"]
            centre = (lat >= s) & (lat <= n) & (lon >= w) & (lon <= e)
            corner = ((clat >= s) & (clat <= n) & (clon >= w) & (clon <= e)).reshape(n_file, -1).any(axis=1)
            # time of every sounding
            tp = spec["time"]
            tv = var(tp["path"])
            traw = read(tp["path"], tv)
            added = {}
            if tp["kind"] == "delta":
                mt = re.search(r"since (\d{4})-(\d{2})-(\d{2})[T ](\d{2}):(\d{2}):(\d{2})", str(tv.getncattr("units")))
                if not mt:
                    raise ValueError(f"cannot read the epoch of {tp['path']} from its units")
                epoch = (datetime(*[int(x) for x in mt.groups()]) - datetime(1970, 1, 1)).total_seconds()
                tsec = epoch + traw.astype("f8") / 1000.0
            else:
                tsec = traw.astype("f8")
            # Damaged soundings. A provider's file can hold a block of rows whose columns no longer belong together (Caltech's
            # TROPO_SIF_2020-02-13_ungridded.nc: 109,267 contiguous rows with a time in 2009-2010 and a centre about 25 degrees from the
            # sounding's own corners). Such a sounding cannot be placed, so it is not stored; they are counted and reported.
            bad_time = (np.zeros(n_file, bool) if day0 is None else
                        (~np.isfinite(tsec)) | (tsec < day0 - 86400.0) | (tsec > day0 + 2 * 86400.0))
            bad_pos = np.zeros(n_file, bool)
            cl_all, co_all = clat.reshape(n_file, -1), clon.reshape(n_file, -1)
            la_all, lo_all = lat.astype("f8"), lon.astype("f8")
            for a in range(0, n_file, 500_000):
                b = min(n_file, a + 500_000)
                cl, co = cl_all[a:b].astype("f8"), co_all[a:b].astype("f8")
                decidable = np.all(np.isfinite(cl) & np.isfinite(co) & (np.abs(cl) <= 90) & (np.abs(co) <= 180), axis=1)
                dlon = (lo_all[a:b, None] - co + 180.0) % 360.0 - 180.0
                with np.errstate(invalid="ignore"):
                    dist = np.sqrt((la_all[a:b, None] - cl) ** 2 + (dlon * np.cos(np.radians(la_all[a:b]))[:, None]) ** 2)
                far = ~(np.where(np.isfinite(dist), dist, np.inf).max(axis=1) <= spec["pos_tol"])
                bad_pos[a:b] = decidable & far                   # a footprint with a missing corner cannot be tested
            damaged = bad_time | bad_pos
            wanted = centre | corner
            sel = np.flatnonzero(wanted & ~damaged)
            sel = sel[np.argsort(tsec[sel], kind="stable")]      # time order; ties keep the file's order
            m = int(sel.size)
            if tp["kind"] == "delta":
                added["sounding_time"] = tsec[sel]

            in_poly = np.zeros(m, "i1")
            fp_poly = np.full(m, -1, "i1")                       # -1: a corner is missing, so not decidable
            if m:
                la, lo, tt = lat[sel].astype("f8"), lon[sel].astype("f8"), tsec[sel]
                if not (np.all(np.isfinite(la)) and np.all(np.isfinite(lo)) and np.all(np.isfinite(tt))
                        and np.all(np.abs(la) <= 90) and np.all(np.abs(lo) <= 180) and np.all(tt > 0)):
                    raise ValueError("a kept sounding has no usable position or time")
                roi = shapely.from_wkb(roi_wkb)
                shapely.prepare(roi)
                in_poly = shapely.contains_xy(roi, lo, la).astype("i1")
                cla, clo = clat[sel].reshape(m, -1).astype("f8"), clon[sel].reshape(m, -1).astype("f8")
                good = np.all(np.isfinite(cla) & np.isfinite(clo) & (np.abs(cla) <= 90) & (np.abs(clo) <= 180), axis=1)
                if good.any():
                    # the hull of the corners is the footprint whatever order the provider lists them in
                    pts = shapely.multipoints(np.stack([clo[good], cla[good]], axis=-1))
                    fp_poly[good] = shapely.intersects(roi, shapely.convex_hull(pts)).astype("i1")

            for p, v in walk(ds):
                attrs = {a: plain(v.getncattr(a)) for a in v.ncattrs()}
                if float(attrs.get("scale_factor", 1)) != 1.0 or float(attrs.get("add_offset", 0)) != 0.0:
                    raise ValueError(f"{p} is packed (scale_factor/add_offset), which this notebook does not expect")
                if v.dtype is str:
                    raise ValueError(f"{p} is a text variable, which this notebook does not expect")
                dims = tuple(v.dimensions)
                if sd in dims:
                    pos = dims.index(sd)
                    a = read(p, v)
                    if a.shape[pos] != n_file:
                        raise ValueError(f"{p} has {a.shape[pos]} rows, the file has {n_file} soundings")
                    a = np.ascontiguousarray(np.moveaxis(np.take(a, sel, axis=pos), pos, 0))
                    rest = [d for d in dims if d != sd]
                    meta[p] = dict(kind="sounding", dtype=v.dtype.str, attrs=attrs, dims=[(d, dim_sizes[d]) for d in rest],
                                   source_dims=list(dims))
                    sounding[p] = a
                else:
                    a = read(p, v)
                    keep = [d for d in dims if dim_sizes[d] != 1]
                    a = a.reshape([dim_sizes[d] for d in keep]) if dims else a
                    meta[p] = dict(kind="day", dtype=v.dtype.str, attrs=attrs, dims=[(d, dim_sizes[d]) for d in keep],
                                   source_dims=list(dims))
                    daily[p] = a
        out.update(ok=True, n_file=int(n_file), n_kept=m, n_centre_in_bbox=int(centre.sum()), n_in_polygon=int(in_poly.sum()),
                   meta=meta, sounding=sounding, day=daily, globals=glob, day_json=day_json, added=added,
                   time_min=float(tsec[sel].min()) if m else None, time_max=float(tsec[sel].max()) if m else None,
                   source_index=sel.astype("i4"), in_polygon=in_poly, footprint_in_polygon=fp_poly,
                   n_damaged=int(damaged.sum()), n_damaged_wanted=int((wanted & damaged).sum()), n_bad_time=int(bad_time.sum()),
                   n_bad_position=int(bad_pos.sum()),
                   damaged_first_row=int(np.flatnonzero(damaged)[0]) if damaged.any() else None,
                   damaged_last_row=int(np.flatnonzero(damaged)[-1]) if damaged.any() else None)
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out


def worker_spec(product, roi):
    spec = PRODUCTS[product]
    return dict(sound_dim=spec["sound_dim"], lat=spec["lat"], lon=spec["lon"], clat=spec["clat"], clon=spec["clon"],
                time=spec["time"], bbox=tuple(roi["bbox"]), globals_skip=list(spec["globals_skip"]),
                day_json_group=spec["day_json_group"], pos_tol=float(POSITION_TOL_DEG))


def start_cluster():
    dask.config.set({"temporary-directory": DASK_TMP, "distributed.worker.daemon": False})
    cluster = LocalCluster(n_workers=DASK_WORKERS, threads_per_worker=1, processes=True,
                           memory_limit=f"{WORKER_MEMORY_GB}GB", dashboard_address=None)
    client = Client(cluster)
    log.info("dask: %d workers x %.1f GB", DASK_WORKERS, WORKER_MEMORY_GB)
    return cluster, client


def extract_group(client, product, items_by_key, grans, roi_future, spec, raw_dir=None):
    """Run extract_file on a batch. A file that does not decode is downloaded once more (its whole item); if it fails
    again the run stops (a day is never skipped silently). Returns the results in the order of `grans`."""
    with timed("verify + extract"):
        def day_start(g):
            return (datetime.combine(g["day"], datetime.min.time()) - datetime(1970, 1, 1)).total_seconds()

        def run(batch):
            futs = [client.submit(extract_file, os.path.join(raw_dir or RAW_DIR, g["name"]), spec, roi_future, day_start(g), pure=False)
                    for g in batch]
            return client.gather(futs)

        results = run(grans)
        bad = [i for i, r in enumerate(results) if not r["ok"]]
        if bad:
            for key in sorted({grans[i]["item"] for i in bad}):
                log.warning("%s: a daily file did not pass the decode check (%s) - downloading the item again", key,
                            next(results[i]["error"] for i in bad if grans[i]["item"] == key))
                fetch_item(product, items_by_key[key], raw_dir)
            for i, r in zip(bad, run([grans[i] for i in bad])):
                if not r["ok"]:
                    raise PipelineStop(f"{grans[i]['name']} is unreadable after two downloads ({r['error']}). "
                                       f"Nothing is lost - re-run to resume; if it persists the file is damaged at the provider.")
                results[i] = r
    return results

In [ ]:
# ═══════════════ CELL 7: the CF output file (create, add variables, append a batch, read back) ═══════════════
# Layout of one output file (one per product). Two record axes, each with its own unlimited dimension:
#   sounding : one row per sounding kept (the bounding box of the aquifer), in time order
#   day      : one row per daily file processed, including days with no sounding over the aquifer.
#              Holds the provider's per-file tables and the ledger: file, item, size, counts.
# Every one of the provider's variables keeps its stored type and its stored values, missing-value codes included.
_COMP = dict(zlib=True, complevel=COMPRESS_LEVEL, shuffle=True)


def raw(ds):
    """Never let netCDF4 mask or scale: this notebook only moves stored values."""
    ds.set_auto_maskandscale(False)
    return ds


def _var(ds, name, dtype, dims, fill=None, chunk=None, **attrs):
    if dtype is str:
        v = ds.createVariable(name, str, dims, chunksizes=chunk)
    else:
        v = ds.createVariable(name, dtype, dims, fill_value=fill, chunksizes=chunk, **_COMP)
    v.setncatts({k: a for k, a in attrs.items() if a is not None})
    return v


def coords_attr(product):
    spec = PRODUCTS[product]
    t = NAME_ADDED if spec["time"]["kind"] == "delta" else nc_name(spec, spec["time"]["path"])
    return f"{t} {nc_name(spec, spec['lat'])} {nc_name(spec, spec['lon'])}"


DAMAGED_RULE = ("A sounding of the provider's daily file is not stored when its time lies more than a day before or two days after the "
                "start of the file's day, or when its centre is farther than %.1f degrees from one of its own footprint corners: the columns of "
                "such a row do not belong together, so it cannot be placed in time or space. day_n_damaged_soundings counts them for every day "
                "(all rows of the file, not only those inside the bounding box); damaged_soundings.jsonl next to the file lists them. "
                "Everything else is stored as the provider's file holds it.")


def ensure_damage_columns(ds):
    """The day column and the global attribute for damaged soundings. Files made before this existed get them on the next write;
    the days stored earlier were not tested and hold the fill value -1 there."""
    if "day_n_damaged_soundings" not in ds.variables:
        _var(ds, "day_n_damaged_soundings", "i4", ("day",), fill=np.int32(-1), chunk=(512,), units="1",
             long_name="Soundings of the provider's daily file that are damaged (time outside the day, or centre far from its own corners) and not stored",
             comment="-1: the day was stored before this test existed and was not assessed")
    if "damaged_sounding_rule" not in ds.ncattrs():
        ds.setncattr("damaged_sounding_rule", DAMAGED_RULE % POSITION_TOL_DEG)


def create_output(path, product, roi):
    spec = PRODUCTS[product]
    w, s, e, n = roi["bbox"]
    ds = raw(nc4.Dataset(path, "w", format="NETCDF4"))
    for d in ("sounding", "day"):
        ds.createDimension(d, None)
    CA = coords_attr(product)
    ds.setncatts(dict(
        Conventions=CF_VERSION, featureType="point", product_kind="tropomi_sif", product=product,
        title=f"{spec['label']}: solar-induced fluorescence soundings over the bounding box of the {ROI_LABEL} (High Plains) aquifer",
        summary="Every sounding of the provider's daily files whose centre or any footprint corner lies in the bounding box of "
                "the aquifer, with every variable the provider publishes, at native footprint resolution. No gridding, "
                "averaging, gap filling, scaling, quality filtering or masking: each value is the value stored in the "
                "provider's file. in_polygon and footprint_in_polygon say where a sounding lies relative to the aquifer polygon.",
        institution=spec["institution"], references=spec["references"], platform=spec["platform"],
        source=("Daily files from " + spec["base"]) if spec["source"] == "sron" else
               f"Monthly archives of daily files from ftp://{spec['host']}{spec['folder']}",
        selection_rule="centre inside the bounding box, or at least one valid footprint corner inside it",
        roi_name=f"{ROI_LABEL} / High Plains aquifer (hp_bound2010)", roi_pad_degrees=float(ROI_PAD_DEG),
        geospatial_lat_min=float(s), geospatial_lat_max=float(n), geospatial_lon_min=float(w), geospatial_lon_max=float(e),
        variable_naming="A variable of the provider's file is stored under its path with the group folded into the name "
                        "(PRODUCT/ and PRODUCT/SUPPORT_DATA/ left out); source_variable holds the original path. Variables "
                        "starting with day_, and day_index, source_index, in_polygon, footprint_in_polygon and sounding_time, are added here.",
        axis_order="Where the provider stores the sounding axis last (the Caltech corner arrays), it is moved to the front; "
                   "nothing else about those values changes. source_dimensions says what the provider's order was.",
        missing_data="Each variable keeps the provider's missing-value code as _FillValue. Rows of a day whose file lacks a "
                     "variable hold _FillValue.",
        history=f"{iso(utcnow())} created by TROPOMI_SIF_Ogallala.ipynb", date_created=iso(utcnow()),
        processing_status="in_progress", committed_soundings=np.int64(0), committed_days=np.int32(0)))
    c1 = (SOUNDING_CHUNK,)
    _var(ds, "day_index", "i4", ("sounding",), chunk=c1, long_name="Row of the day axis this sounding comes from")
    _var(ds, "source_index", "i4", ("sounding",), chunk=c1,
         long_name="Position of this sounding on the sounding axis of the provider's daily file (zero-based)")
    _var(ds, "in_polygon", "i1", ("sounding",), fill=np.int8(-127), chunk=c1,
         long_name="Centre of the sounding lies inside the aquifer polygon", flag_values=np.array([0, 1], "i1"),
         flag_meanings="outside inside", coordinates=CA)
    _var(ds, "footprint_in_polygon", "i1", ("sounding",), fill=np.int8(-127), chunk=c1,
         long_name="Footprint (hull of the corners) intersects the aquifer polygon", flag_values=np.array([-1, 0, 1], "i1"),
         flag_meanings="corner_missing no yes", coordinates=CA)
    if spec["time"]["kind"] == "delta":
        _var(ds, NAME_ADDED, "f8", ("sounding",), fill=np.float64(-1.0), chunk=c1, standard_name="time", axis="T",
             units="seconds since 1970-01-01 00:00:00", calendar="standard", long_name="UTC time of the sounding (computed here)",
             comment=DELTA_COMMENT)
    cd = (512,)
    _var(ds, "day_time", "i4", ("day",), chunk=cd, long_name="Date of the provider's daily file", standard_name="time",
         units="days since 1970-01-01 00:00:00", calendar="standard")
    for name, text in (("day_source_file", "Name of the provider's daily file"),
                       ("day_source_item", "File or archive that was downloaded (the same as the daily file for TROPOSIF, the monthly archive for Caltech)"),
                       ("day_source_etag", "ETag the server sent with the daily file (empty when the server sends none)"),
                       ("day_source_last_modified", "Last-Modified date of the daily file (TROPOSIF) or modification time of the archive (Caltech, UTC)"),
                       ("day_item_last_modified", "Last-Modified / modification time of the downloaded item")):
        _var(ds, name, str, ("day",), chunk=cd, long_name=text)
    _var(ds, "day_source_size", "i8", ("day",), chunk=cd, long_name="Size of the daily file", units="byte")
    _var(ds, "day_item_size", "i8", ("day",), chunk=cd, long_name="Size of the downloaded item (file or archive)", units="byte")
    for name, text in (("day_n_soundings_file", "Soundings in the provider's daily file (whole globe)"),
                       ("day_n_soundings", "Soundings of the day kept here"),
                       ("day_n_in_polygon", "Soundings of the day whose centre is inside the aquifer polygon")):
        _var(ds, name, "i4", ("day",), chunk=cd, long_name=text, units="1")
    _var(ds, "day_first_sounding", "i8", ("day",), chunk=cd, long_name="Row of the sounding axis where the soundings of this day start")
    ensure_damage_columns(ds)
    for a in spec["globals_per_day"]:
        _var(ds, "day_global_" + a, str, ("day",), chunk=cd, long_name=f"Attribute {a} of the provider's daily file")
    if spec["day_json_group"]:
        _var(ds, "day_retrieval_settings", str, ("day",), chunk=cd,
             long_name=f"Attributes of the group {spec['day_json_group']} of the daily file, as JSON")
    ds.sync()
    return ds


def _axis_of(v):
    return v.dimensions[0] if v.dimensions else None


def _default_fill(dt):
    return np.array(nc4.default_fillvals[dt.str[1:]], dtype=dt)[()]


def ensure_var(ds, product, path, m, source_file):
    """Create the output variable for one of the provider's variables on first sight, in its stored type, with its
    missing-value code as _FillValue. On later sight: the type and shape must be the same."""
    spec = PRODUCTS[product]
    name = nc_name(spec, path)
    axis = m["kind"]
    dims = (axis,) + tuple(d for d, _ in m["dims"])
    dt = np.dtype(m["dtype"])
    a = m["attrs"]
    code = None
    for key in ("_FillValue", "missing_value"):
        if a.get(key) is not None:
            code = np.array(a[key], dtype=dt)[()]
            break
    if name in ds.variables:
        v = ds[name]
        have = np.dtype(v.dtype).str
        if have != m["dtype"] or tuple(v.dimensions) != dims:
            raise DataIntegrityError(f"{path} in {source_file} is {m['dtype']} {dims}; the output holds it as {have} "
                                     f"{tuple(v.dimensions)}. The provider changed the variable - not merged.")
        if v.source_variable[1:] != path:
            raise DataIntegrityError(f"{path} and {v.source_variable[1:]} would both be stored as {name}.")
        return v
    for d, size in m["dims"]:
        if d not in ds.dimensions:
            ds.createDimension(d, size)
        elif ds.dimensions[d].size != size:
            raise DataIntegrityError(f"{path} in {source_file}: axis {d} has {size} entries, the output has {ds.dimensions[d].size}.")
    tail = tuple(size for _, size in m["dims"])
    cells = int(np.prod(tail)) if tail else 1
    if axis == "sounding":
        chunk = (max(1, SOUNDING_CHUNK // cells),) + tail
    else:
        chunk = (max(1, min(512, (1 << 18) // max(1, cells * dt.itemsize))),) + tail
    src_units = a.get("units")
    note = None
    if code is None:
        fill = _default_fill(dt)
        if spec["undeclared_fill"] and dt.kind == "f":
            note = (f"The provider's file declares no missing-value code, and the netCDF default {fill:.6g} occurs in some "
                    "variables (for example the 735 nm retrieval of soundings the 735-758 nm window could not fit), so it is "
                    "declared here as _FillValue.")
    else:
        fill = code
    units = spec["units_forced"].get(path) or UNITS_CF.get(src_units)
    flags = None
    if units is None and isinstance(src_units, str) and "=" in src_units and src_units.startswith(("([", "[")):
        pairs = re.findall(r"(\w+)=(-?\d+)", src_units)                  # a class legend in the units text (LC_MASK)
        flags = ([int(v) for _, v in pairs], " ".join(k for k, _ in pairs))
        units = "1"
    if units is None and dt.kind == "f" and src_units in (None, "n/a"):
        units = "1"
    if units is None and src_units not in (None, "n/a") and flags is None:
        units = src_units                                                # an unforeseen unit string: keep the provider's wording
    if path in spec["units_forced"] and src_units is None:
        note = ((note + " ") if note else "") + "The provider's file states no units; they are given here from the variable's name and values."
    attrs = dict(long_name=a.get("long_name") or path.split("/")[-1].replace("_", " "), description=a.get("description"), source_variable="/" + path, comment=note,
                 source_dimensions=", ".join(m["source_dims"]) if axis == "sounding" and m["source_dims"][0] != spec["sound_dim"] else None)
    sname = a.get("standard_name")
    if path == spec["lat"]:
        sname = "latitude"
    elif path == spec["lon"]:
        sname = "longitude"
    if path == spec["time"]["path"]:
        if spec["time"]["kind"] == "delta":
            units, attrs["comment"] = "milliseconds", DELTA_COMMENT
        else:
            attrs["calendar"] = "standard"
            sname = "time"
    if path.endswith("solar_zenith_angle"):
        sname = "solar_zenith_angle"
    elif path.endswith("viewing_zenith_angle"):
        sname = "sensor_zenith_angle"
    attrs.update(units=units, standard_name=sname if sname in STANDARD_NAME_OK else None)
    if flags:
        attrs.update(flag_values=np.array(flags[0], dtype=dt), flag_meanings=flags[1])
    if axis == "sounding" and path not in (spec["lat"], spec["lon"], spec["time"]["path"]):
        attrs["coordinates"] = coords_attr(product)
    v = _var(ds, name, dt, dims, fill=fill, chunk=chunk, **attrs)
    extra = {}
    if src_units is not None and src_units != units:
        extra["source_units"] = src_units
    if a.get("standard_name") is not None and a.get("standard_name") != attrs["standard_name"]:
        extra["source_standard_name"] = a["standard_name"]        # free text in the file, not a CF standard name
    for k in ("valid_range", "valid_min", "valid_max"):
        if k in a:
            extra["source_" + k] = a[k]                               # not valid_*: netCDF4 and xarray would mask on it
    v.setncatts(extra)
    return v


def _fill_block(v, shape):
    return np.full(shape, v.getncattr("_FillValue"), dtype=v.dtype)


def _same(got, want):
    """Bitwise comparison (so NaN equals NaN and -0.0 differs from 0.0)."""
    if isinstance(want, np.ndarray) and want.dtype != object:
        got = np.asarray(got)
        return got.dtype == want.dtype and got.shape == want.shape and got.tobytes() == want.tobytes()
    return [str(x) for x in np.asarray(got, dtype=object).reshape(-1)] == [str(x) for x in np.asarray(want, dtype=object).reshape(-1)]


def write_batch(ds, product, grans, results, state):
    """Append a batch of days (in date order) to the two axes, then read every written block back and compare it
    bitwise with what was meant to be written. Returns the new (soundings, days)."""
    spec = PRODUCTS[product]
    n0, d0 = state["n"], state["d"]
    K = len(results)
    ensure_damage_columns(ds)
    with timed("write"):
        for g, r in zip(grans, results):
            for path, m in r["meta"].items():
                ensure_var(ds, product, path, m, g["name"])
        counts = np.array([r["n_kept"] for r in results], "i8")
        first = n0 + np.concatenate([[0], np.cumsum(counts)[:-1]])
        M = int(counts.sum())
        written = []

        def put(name, lo, block):
            ds[name][lo:lo + len(block)] = block
            written.append((name, lo, block))

        if M:
            own = dict(
                day_index=np.repeat(np.arange(d0, d0 + K, dtype="i4"), counts),
                source_index=np.concatenate([r["source_index"] for r in results]).astype("i4"),
                in_polygon=np.concatenate([r["in_polygon"] for r in results]).astype("i1"),
                footprint_in_polygon=np.concatenate([r["footprint_in_polygon"] for r in results]).astype("i1"))
            if NAME_ADDED in ds.variables:
                own[NAME_ADDED] = np.concatenate([r["added"][NAME_ADDED] for r in results]).astype("f8")
            for name, block in own.items():
                put(name, n0, block)
            for name, v in ds.variables.items():
                if _axis_of(v) != "sounding" or "source_variable" not in v.ncattrs():
                    continue
                path = v.source_variable[1:]
                tail = v.shape[1:]
                block = np.concatenate([r["sounding"][path] if path in r["sounding"] else _fill_block(v, (r["n_kept"],) + tail)
                                        for r in results])
                put(name, n0, block)
            t = np.concatenate([r["added"][NAME_ADDED] if NAME_ADDED in r["added"] else r["sounding"][spec["time"]["path"]].astype("f8")
                                for r in results])
            if np.any(np.diff(t) < 0) or (state.get("last_time") is not None and t[0] < state["last_time"]):
                log.warning("%s: soundings of %s to %s are not in time order across files.", product, grans[0]["day"], grans[-1]["day"])
            state["last_time"] = float(t.max())
        obj = lambda xs: np.array(list(xs), dtype=object)
        put("day_time", d0, np.array([day_number(g["day"]) for g in grans], "i4"))
        put("day_source_file", d0, obj(g["name"] for g in grans))
        put("day_source_item", d0, obj(g["item"] for g in grans))
        put("day_source_etag", d0, obj(g["etag"] for g in grans))
        put("day_source_last_modified", d0, obj(g["last_modified"] for g in grans))
        put("day_item_last_modified", d0, obj(g["item_last_modified"] for g in grans))
        put("day_source_size", d0, np.array([g["size"] for g in grans], "i8"))
        put("day_item_size", d0, np.array([g["item_size"] for g in grans], "i8"))
        put("day_n_soundings_file", d0, np.array([r["n_file"] for r in results], "i4"))
        put("day_n_soundings", d0, counts.astype("i4"))
        put("day_n_in_polygon", d0, np.array([r["n_in_polygon"] for r in results], "i4"))
        put("day_n_damaged_soundings", d0, np.array([r["n_damaged"] for r in results], "i4"))
        put("day_first_sounding", d0, first.astype("i8"))
        for a in spec["globals_per_day"]:
            put("day_global_" + a, d0, obj(r["globals"].get(a, "") for r in results))
        if spec["day_json_group"]:
            put("day_retrieval_settings", d0, obj(r["day_json"] or "" for r in results))
        for name, v in ds.variables.items():
            if _axis_of(v) != "day" or "source_variable" not in v.ncattrs():
                continue
            path = v.source_variable[1:]
            tail = v.shape[1:]
            put(name, d0, np.stack([r["day"][path] if path in r["day"] else _fill_block(v, tail) for r in results]))
        if "source_attributes_from" not in ds.ncattrs():          # the file-level attributes, copied once
            ds.setncatts({"source_" + k: val for k, val in results[0]["globals"].items() if k not in spec["globals_per_day"]})
            ds.source_attributes_from = grans[0]["name"]
        ds.sync()
    with timed("read-back check"):
        for name, lo, block in written:
            if not _same(ds[name][lo:lo + len(block)], block):
                raise DataIntegrityError(f"{name}: rows {lo} to {lo + len(block) - 1} read back differently from what was written. "
                                         f"The output on scratch is not trusted; the copy on Drive is untouched.")
    return n0 + M, d0 + K


def commit(ds, n, d, complete):
    """Move the commit markers. Rows beyond them are never read by this notebook."""
    ds.committed_soundings, ds.committed_days = np.int64(n), np.int32(d)
    ds.date_modified = iso(utcnow())
    if d:
        t = ds["day_time"][:d]
        ds.time_coverage_start = (DAY_EPOCH + timedelta(days=int(t[0]))).isoformat()
        ds.time_coverage_end = (DAY_EPOCH + timedelta(days=int(t[-1]))).isoformat()
    ds.processing_status = "complete" if complete else "in_progress"
    ds.sync()


def read_state(path):
    """What the output already holds: commit markers and the item stored for each day."""
    with nc4.Dataset(path) as ds:
        raw(ds)
        if str(getattr(ds, "product_kind", "")) != "tropomi_sif":
            raise PipelineStop(f"{path} is not an output of this notebook.")
        d = min(int(ds.committed_days), ds.dimensions["day"].size)
        return dict(product=str(ds.product), d=d, n=int(ds.committed_soundings),
                    bbox=(float(ds.geospatial_lon_min), float(ds.geospatial_lat_min), float(ds.geospatial_lon_max), float(ds.geospatial_lat_max)),
                    items=[str(x) for x in ds["day_source_item"][:d]], files=[str(x) for x in ds["day_source_file"][:d]],
                    item_sizes=[int(x) for x in ds["day_item_size"][:d]], item_mtimes=[str(x) for x in ds["day_item_last_modified"][:d]],
                    days=[DAY_EPOCH + timedelta(days=int(x)) for x in ds["day_time"][:d]],
                    first_sounding=np.asarray(ds["day_first_sounding"][:d], "i8"),
                    sizes=dict(sounding=ds.dimensions["sounding"].size, day=ds.dimensions["day"].size))


def _committed_days(path):
    try:
        with nc4.Dataset(path) as ds:
            return int(ds.committed_days)
    except Exception:
        return -1


def backup(local, drive_path):
    """Verified copy of the output to Drive (temporary name, MD5, then rename)."""
    robust_drive_copy(local, drive_path)


def stage_output(name):
    """Put the output to continue from on scratch: the copy on Drive, unless scratch already holds a
    readable copy that is further along (a run interrupted before its next backup)."""
    local, on_drive = os.path.join(NC_DIR, name), os.path.join(DRIVE_OUT, name)
    if REBUILD:
        if os.path.exists(local):
            os.remove(local)
        log.info("REBUILD: %s starts again (the copy on Drive is replaced at the first backup).", name)
        return local
    have_local = _committed_days(local) if os.path.exists(local) else -1
    have_drive = _committed_days(on_drive) if os.path.exists(on_drive) else -1
    if os.path.exists(on_drive) and have_drive < 0:
        raise PipelineStop(f"{on_drive} exists but cannot be read. Move it away, or set REBUILD = True.")
    if have_drive >= 0 and have_drive >= have_local:
        restore_from_drive(on_drive, local)
    elif have_local >= 0:
        log.info("Continuing from the copy on scratch (%d days; Drive has %d).", have_local, max(have_drive, 0))
    elif os.path.exists(local):
        os.remove(local)                                         # unreadable leftover
    return local

In [ ]:
# ═══════════════ CELL 8: one product, batch by batch, then RUN ═══════════════
def resume_point(st, items):
    """Where to continue. The stored items are kept while they are, in order, the items the provider lists today and
    unchanged (same size and modification time). Returns (items to keep, index in `items` to continue from)."""
    stored = list(dict.fromkeys(st["items"]))                    # items in order of first use (each is stored whole)
    keys = [it["key"] for it in items]
    k = 0
    while k < len(stored) and k < len(keys) and stored[k] == keys[k]:
        k += 1
    if k < len(stored):
        if k == len(keys):
            log.info("The file holds %d item(s) beyond the date window of this run; nothing is removed.", len(stored) - k)
            return len(stored), len(keys)
        log.warning("The provider's list differs from the stored items from %s on (stored %s, listed %s).", stored[k], stored[k], keys[k])
        if not REPROCESS_CHANGED:
            log.warning("The stored days are left as they are. Set REPROCESS_CHANGED = True to rebuild from there, or REBUILD = True.")
            later = next((i for i, key in enumerate(keys) if key > stored[-1]), len(keys))
            return len(stored), later
        return k, k
    first_day = {key: st["items"].index(key) for key in stored}
    changed = [i for i, key in enumerate(stored)
               if st["item_sizes"][first_day[key]] != items[i]["size"] or st["item_mtimes"][first_day[key]] != items[i]["last_modified"]]
    if changed:
        if REPROCESS_CHANGED:
            log.warning("The provider has replaced %d stored item(s), the first %s; rebuilding from there.", len(changed), stored[changed[0]])
            return changed[0], changed[0]
        log.warning("The provider has replaced %d stored item(s) (size or modification time differs), the first %s. The stored "
                    "values are kept; set REPROCESS_CHANGED = True to rebuild from there.", len(changed), stored[changed[0]])
    return len(stored), len(stored)


def run_product(client, roi, roi_future, product):
    """Download a batch of items, verify, keep the aquifer's soundings, append, read back, delete, repeat."""
    spec = PRODUCTS[product]
    items, lo, hi = select_items(product)
    by_key = {it["key"]: it for it in items}
    log.info("%s: %d item(s) selected (%s to %s), %s to download", product, len(items), items[0]["name"], items[-1]["name"],
             human(sum(it["size"] for it in items)))
    if spec["source"] == "sron":
        gaps = calendar_gaps([it["day"] for it in items])
        log.info("%s: %d calendar day(s) have no file at the provider (longest gap %s)", product, sum(g[2] for g in gaps),
                 ("%s to %s" % max(gaps, key=lambda g: g[2])[:2]) if gaps else "none")
    name = spec["out_name"]
    on_drive = os.path.join(DRIVE_OUT, name)
    local = stage_output(name)
    n = d = start = 0
    if os.path.exists(local):
        st = read_state(local)
        if st["product"] != product:
            raise PipelineStop(f"{local} was built for {st['product']}, not {product}.")
        if np.max(np.abs(np.array(st["bbox"]) - np.array(roi["bbox"]))) > 1e-9:
            raise PipelineStop("The file on Drive was built for a different bounding box. Restore ROI_PAD_DEG and the shapefile, or set REBUILD = True.")
        keep_items, start = resume_point(st, items)
        stored = list(dict.fromkeys(st["items"]))
        d = sum(1 for key in st["items"] if key in set(stored[:keep_items]))
        n = st["n"] if d == st["d"] else int(st["first_sounding"][d])
    todo = items[start:]
    if not todo:
        log.info("%s: the file already holds every selected item (%d days, %d soundings). Nothing to do.", product, d, n)
        return dict(path=on_drive, days=d, new_days=0, soundings=n)
    log.info("%s: keeping %d day(s) / %d soundings, processing %d item(s) from %s", product, d, n, len(todo), todo[0]["name"])
    ds = create_output(local, product, roi) if not os.path.exists(local) else raw(nc4.Dataset(local, "a"))
    state = dict(n=n, d=d, last_time=None)
    if n and NAME_ADDED in ds.variables:
        state["last_time"] = float(ds[NAME_ADDED][n - 1])
    elif n:
        state["last_time"] = float(ds[nc_name(spec, spec["time"]["path"])][n - 1])
    commit(ds, n, d, False)                              # anything after these markers is being rewritten
    d_start, n_start, t_run, last_backup = d, n, time.time(), time.time()
    trusted = True
    nb = BATCH_ITEMS[product]
    try:
        for b0 in range(0, len(todo), nb):
            batch = todo[b0:b0 + nb]
            grans = fetch_batch(product, batch)
            results = extract_group(client, product, by_key, grans, roi_future, worker_spec(product, roi))
            damage = []
            for g, r in zip(grans, results):             # damaged soundings are left out, never silently
                if r["n_damaged"]:
                    share = r["n_damaged"] / max(1, r["n_file"])
                    damage.append(dict(file=g["name"], rows_in_file=r["n_file"], damaged=r["n_damaged"], share=round(share, 5),
                                       bad_time=r["n_bad_time"], bad_position=r["n_bad_position"], first_row=r["damaged_first_row"],
                                       last_row=r["damaged_last_row"], would_have_been_kept=r["n_damaged_wanted"]))
                    log.warning("%s: %s of %s soundings (%.1f %%) are damaged in the provider's file, rows %s to %s (%s with a time outside the "
                                "day, %s with a centre more than %.1f degrees from their own corners); %s of them lie in the bounding box. "
                                "They are not stored.", g["name"], f"{r['n_damaged']:,}", f"{r['n_file']:,}", 100 * share,
                                f"{r['damaged_first_row']:,}", f"{r['damaged_last_row']:,}", f"{r['n_bad_time']:,}", f"{r['n_bad_position']:,}",
                                POSITION_TOL_DEG, f"{r['n_damaged_wanted']:,}")
                    if share > MAX_DAMAGED_FRACTION:
                        log_events("damaged_soundings.jsonl", damage)
                        raise DataIntegrityError(f"{g['name']}: {share:.0%} of its soundings are damaged (more than MAX_DAMAGED_FRACTION = "
                                                 f"{MAX_DAMAGED_FRACTION:.0%}); the file looks mislabelled or broken at the provider.")
            log_events("damaged_soundings.jsonl", damage)
            for g, r in zip(grans, results):             # a file whose soundings are days away from its own date is not stored
                if r["time_min"] is not None:
                    t0 = (datetime.combine(g["day"], datetime.min.time()) - datetime(1970, 1, 1)).total_seconds()
                    if r["time_min"] < t0 - 86400 or r["time_max"] > t0 + 2 * 86400:
                        raise DataIntegrityError(f"{g['name']} holds soundings from {datetime.utcfromtimestamp(r['time_min'])} to "
                                                 f"{datetime.utcfromtimestamp(r['time_max'])}, not the day of its name.")
            n, d = write_batch(ds, product, grans, results, state)
            state.update(n=n, d=d)
            final = b0 + nb >= len(todo)
            commit(ds, n, d, final)
            remove_raw([g["name"] for g in grans])
            TOTALS[product + "_files"] = TOTALS.get(product + "_files", 0) + len(grans)
            log.info("%s %s to %s: %d daily files -> %d soundings kept (%d inside the polygon, %d file(s) with none) | total %d days, "
                     "%d soundings | %.1f GB free", product, grans[0]["day"], grans[-1]["day"], len(grans), sum(r["n_kept"] for r in results),
                     sum(r["n_in_polygon"] for r in results), sum(1 for r in results if r["n_kept"] == 0), d, n, free_gb())
            del results
            if not final and time.time() - last_backup >= BACKUP_INTERVAL_S:
                ds.close()
                ds = None
                backup(local, on_drive)
                ds = raw(nc4.Dataset(local, "a"))
                last_backup = time.time()
    except DataIntegrityError:
        trusted = False
        raise
    except (KeyboardInterrupt, PipelineStop) as e:
        log.warning("%s stopped (%s). Saving the %d committed day(s) to Drive.", product, type(e).__name__, d)
        raise
    finally:
        if ds is not None:
            ds.close()
        if not trusted:
            os.remove(local)                            # the next run starts from the copy on Drive
        elif os.path.exists(local) and d > d_start:
            backup(local, on_drive)
    sizes = read_state(local)["sizes"]
    if sizes["sounding"] > n or sizes["day"] > d:
        log.warning("%s: the file holds %d sounding row(s) after the last committed one (left over from a rebuild of changed "
                    "days). Readers must stop at committed_soundings; set REBUILD = True to write a file without them.", product, sizes["sounding"] - n)
    return dict(path=on_drive, days=d, new_days=d - d_start, soundings=n, new_soundings=n - n_start, seconds=time.time() - t_run,
                run_bytes=sum(it["size"] for it in todo), run_items=len(todo))


def run_all():
    """Run every product. Stops cleanly on Ctrl-C / interrupt: whatever is committed is copied to Drive."""
    TIMES.clear()
    TOTALS.clear()
    t0 = time.time()
    roi = load_roi()
    ensure_disk_free("start")
    cluster, client = start_cluster()
    out = {}
    try:
        roi_future = client.scatter(roi["wkb"], broadcast=True)      # the polygon travels to the workers once
        for product in PRODUCTS_TO_RUN:
            out[product] = run_product(client, roi, roi_future, product)
    except KeyboardInterrupt:
        log.warning("Interrupted. Committed data are on Drive; Run all again to continue.")
    except PipelineStop as e:
        log.error("STOPPED: %s", e)
    except DataIntegrityError as e:
        log.error("STOPPED, nothing was copied to Drive from the failed batch: %s", e)
    finally:
        client.close()
        cluster.close()
        shutil.rmtree(DASK_TMP, ignore_errors=True)
    log.info("Run time %.1f min | by stage (s): %s | downloaded %s", (time.time() - t0) / 60,
             {k: round(v) for k, v in TIMES.items()}, human(TOTALS.get("bytes_downloaded", 0)))
    if TEST_MODE:
        for product, r in out.items():
            if r.get("seconds") and r.get("new_days"):
                try:
                    whole = select_items(product, whole_record=True)[0]
                except Exception as e:
                    log.warning("%s: the whole record could not be listed for the projection (%s)", product, e)
                    continue
                total = sum(it["size"] for it in whole) if all("size" in it for it in whole) else len(whole) * r["run_bytes"] / max(1, r["run_items"])
                scale = total / max(1, r["run_bytes"])
                size = os.path.getsize(r["path"])
                log.info("%s rehearsal: %d days, %d soundings, output %s. Projected whole record (%d items, about %s to download): about "
                         "%.1f h at this speed, output about %s.", product, r["new_days"], r["soundings"], human(size), len(whole),
                         human(total), r["seconds"] * scale / 3600, human(size * scale))
        log.info("Rehearsal finished. Set TEST_MODE = False in Cell 2 and Run all for the whole record.")
    return out


RESULT = run_all()

In [ ]:
# ═══════════════ CELL 9: audit of the files on Drive + independent re-download check ═══════════════
def audit_output(path):
    """Reopen the copy on Drive and check structure, bookkeeping and order. Returns (info, problems)."""
    problems, info = [], {}
    with nc4.Dataset(path) as ds:
        raw(ds)
        product = str(ds.product)
        spec = PRODUCTS[product]
        n, d = int(ds.committed_soundings), int(ds.committed_days)
        sizes = {k: ds.dimensions[k].size for k in ("sounding", "day")}
        if (sizes["sounding"], sizes["day"]) != (n, d):
            problems.append(f"rows beyond the commit markers: axes hold {sizes}, committed are {n} soundings / {d} days")
        cnt = np.asarray(ds["day_n_soundings"][:d], "i8")
        first = np.asarray(ds["day_first_sounding"][:d], "i8")
        if d and (first[0] != 0 or np.any(first[1:] != first[:-1] + cnt[:-1]) or first[-1] + cnt[-1] != n):
            problems.append("the day ledger does not add up to the sounding axis")
        if n:
            di = np.asarray(ds["day_index"][:n], "i8")
            if di.min() < 0 or di.max() >= d or np.any(np.bincount(di, minlength=d) != cnt):
                problems.append("day_index does not agree with the day ledger")
            if np.any(np.diff(di) < 0):
                problems.append("soundings are not grouped by day")
        days = [DAY_EPOCH + timedelta(days=int(x)) for x in ds["day_time"][:d]]
        if any(b <= a for a, b in zip(days, days[1:])):
            problems.append("the day axis is not strictly increasing")
        files = [str(x) for x in ds["day_source_file"][:d]]
        if len(set(files)) != len(files):
            problems.append("a daily file is stored twice")
        svars = [k for k, v in ds.variables.items() if _axis_of(v) == "sounding" and "source_variable" in v.ncattrs()]
        dvars = [k for k, v in ds.variables.items() if _axis_of(v) == "day" and "source_variable" in v.ncattrs()]
        for k, v in ds.variables.items():
            if "long_name" not in v.ncattrs():
                problems.append(f"{k}: no long_name")
            if "_FillValue" not in v.ncattrs() and v.dtype is not str and v.dtype.kind == "f":
                problems.append(f"{k}: no _FillValue")
        order = "n/a"
        if n:
            tname = NAME_ADDED if NAME_ADDED in ds.variables else nc_name(spec, spec["time"]["path"])
            t = np.asarray(ds[tname][:n])
            order = "non-decreasing" if np.all(np.diff(t) >= 0) else "NOT in time order"
            if order != "non-decreasing":
                problems.append("soundings are not in time order")
            la, lo = np.asarray(ds[nc_name(spec, spec["lat"])][:n]), np.asarray(ds[nc_name(spec, spec["lon"])][:n])
            w, s, e, nn = (float(getattr(ds, k)) for k in ("geospatial_lon_min", "geospatial_lat_min", "geospatial_lon_max", "geospatial_lat_max"))
            near = (la >= s - 1) & (la <= nn + 1) & (lo >= w - 1) & (lo <= e + 1)
            if not near.all():
                problems.append(f"{int((~near).sum())} sounding(s) lie more than 1 degree from the bounding box")
        empty, at_fill = [], {}
        for k in svars:
            v = ds[k]
            if n and v.dtype.kind in "fiu":
                fill = v.getncattr("_FillValue")
                share = float(np.mean(np.asarray(v[:n]) == fill)) if n < 2_000_000 else float(np.mean(np.asarray(v[:2_000_000]) == fill))
                if share == 1.0:
                    empty.append(k)
                elif share > 0:
                    at_fill[k] = round(100 * share, 3)

        def hist(name):
            if not n or name not in ds.variables:
                return {}
            u, c = np.unique(ds[name][:n], return_counts=True)
            return {int(a): int(b) for a, b in zip(u, c)}

        if "day_n_damaged_soundings" in ds.variables:
            dm = np.asarray(ds["day_n_damaged_soundings"][:d], "i8")
            info.update(days_with_damaged_soundings=int((dm > 0).sum()), damaged_soundings_not_stored=int(dm[dm > 0].sum()),
                        days_not_assessed_for_damage=int((dm < 0).sum()))
        gaps = calendar_gaps(days)
        info.update(product=product, soundings=n, days=d, first=str(days[0]) if d else None, last=str(days[-1]) if d else None,
                    size=os.path.getsize(path), sounding_variables=len(svars), day_variables=len(dvars),
                    days_without_soundings=int((cnt == 0).sum()), time_order=order,
                    in_polygon=hist("in_polygon").get(1, 0), footprint_in_polygon=hist("footprint_in_polygon").get(1, 0),
                    calendar_days_without_file=sum(g[2] for g in gaps),
                    longest_gaps=[f"{a} to {b} ({c} d)" for a, b, c in sorted(gaps, key=lambda g: -g[2])[:5]],
                    variables_all_missing=empty, share_of_soundings_at_fill_percent=at_fill,
                    processing_status=str(getattr(ds, "processing_status", "?")))
    return info, problems


def cross_check(path, k):
    """Download k stored items again, read the provider's variables straight from the fresh files (no worker, no batch
    code) and compare every stored value of the days in those items with them, bit for bit.
    Returns (days checked, values compared)."""
    rng = random.Random(20261022)
    tmp = os.path.join(SCRATCH, "recheck")
    os.makedirs(tmp, exist_ok=True)
    checked = values = 0
    with nc4.Dataset(path) as ds:
        raw(ds)
        product = str(ds.product)
        spec = PRODUCTS[product]
        d = int(ds.committed_days)
        cnt = np.asarray(ds["day_n_soundings"][:d], "i8")
        first = np.asarray(ds["day_first_sounding"][:d], "i8")
        item_of = [str(x) for x in ds["day_source_item"][:d]]
        isize = {key: int(ds["day_item_size"][i]) for i, key in reversed(list(enumerate(item_of)))}
        pool = sorted({key for i, key in enumerate(item_of) if cnt[i] > 0}, key=lambda key: isize[key])[:max(k, 1) * 3]
        picks = rng.sample(pool, min(k, len(pool)))                  # among the smaller items, so the check stays cheap
        for key in sorted(picks):
            it = dict(name=key, key=key, size=isize[key], last_modified="", day=None)
            if spec["source"] == "sron":
                mt = spec["file_re"].search(key)
                it.update(day=date(int(mt.group(1)), int(mt.group(2)), int(mt.group(3))),
                          url=f"{spec['base']}{mt.group(1)}/{mt.group(2)}/{key}")
            days_here = [i for i, kk in enumerate(item_of) if kk == key and cnt[i] > 0]
            want_days = {days_here[len(days_here) // 2]} if spec["source"] == "caltech" else set(days_here)
            if spec["source"] == "sron":
                grans = fetch_item(product, it, tmp)
            else:                                                    # one day out of the archive is enough
                day = DAY_EPOCH + timedelta(days=int(ds["day_time"][min(want_days)]))
                tar = download_ftp(spec, it, tmp)
                grans = extract_tar(spec, it, tar, tmp, day, day)
                os.remove(tar)
            try:
                for g in grans:
                    i = next((j for j in want_days if g["name"] == str(ds["day_source_file"][j])), None)
                    if i is None:
                        continue
                    lo, hi = int(first[i]), int(first[i] + cnt[i])
                    src_idx = np.asarray(ds["source_index"][lo:hi], "i8")
                    with nc4.Dataset(os.path.join(tmp, g["name"])) as src:
                        raw(src)
                        for vname, v in ds.variables.items():
                            if "source_variable" not in v.ncattrs():
                                continue
                            axis = _axis_of(v)
                            sv = src
                            for part in v.source_variable[1:].split("/"):
                                sv = sv.groups[part] if part in getattr(sv, "groups", {}) else sv.variables[part]
                            if axis == "sounding":
                                pos = list(sv.dimensions).index(spec["sound_dim"])
                                want = np.ascontiguousarray(np.moveaxis(np.take(np.asarray(sv[...]), src_idx, axis=pos), pos, 0))
                                got = np.asarray(v[lo:hi])
                            else:
                                want = np.asarray(sv[...])
                                want = want.reshape([s for s in want.shape if s != 1]) if want.ndim else want
                                got = np.asarray(v[i])
                            if not _same(got, want):
                                raise DataIntegrityError(f"RE-CHECK FAILED: {vname} of {g['name']} differs from the provider's file.")
                            values += got.size
                    checked += 1
            finally:
                for g in grans:
                    remove_raw([g["name"]], tmp)
    shutil.rmtree(tmp, ignore_errors=True)
    return checked, values


def audit_all():
    ok = True
    for product in PRODUCTS_TO_RUN:
        path = os.path.join(DRIVE_OUT, PRODUCTS[product]["out_name"])
        if not os.path.exists(path):
            log.warning("AUDIT %s: no file on Drive yet (%s)", product, path)
            ok = False
            continue
        info, problems = audit_output(path)
        log.info("AUDIT %s: %s", os.path.basename(path), json.dumps(info, default=str))
        try:
            c, v = cross_check(path, CROSS_CHECKS[product])
            log.info("AUDIT %s: %d day(s) downloaded again, %d stored values identical to the provider's files", product, c, v)
        except DataIntegrityError as e:
            problems.append(str(e))
        except PipelineStop as e:
            log.warning("AUDIT %s: the re-download check could not run (%s)", product, e)
        for p in problems:
            log.error("AUDIT %s PROBLEM: %s", product, p)
        ok = ok and not problems
        if info.get("processing_status") != "complete":
            log.warning("AUDIT %s: the file is not complete yet - Run all again to continue.", product)
    log.info("AUDIT RESULT: %s", "no problem found" if ok else "PROBLEMS FOUND - see the lines above")
    return ok


AUDIT_OK = audit_all()

# Reading the files (one row per sounding; the footprint corners are the extra axis of *_bounds / *_bnds):
#   import xarray as xr
#   ds = xr.open_dataset("/content/drive/MyDrive/MSUGWB/TROPOMI_SIF/TROPOSIF_L2B_v2.1_Ogallala.nc")
#   ds = ds.isel(sounding=slice(0, int(ds.committed_soundings)))             # stop at the commit marker
#   inside = ds.where(ds.in_polygon == 1, drop=True)                         # centre inside the aquifer polygon
#   sif = inside.SIF_743                                                     # mW m-2 sr-1 nm-1, 743-758 nm window
# xarray turns the missing-value code into NaN (and integer variables with a code into floats); use
# mask_and_scale=False for the stored values. Typical filters: cloud fraction (TROPOSIF INPUT_DATA_cloud_fraction_L2 /
# Caltech cloud_fraction), solar zenith angle; none is applied here.